# A2C on FrozenLake

This notebook adapts the CartPole A2C tutorial to the deterministic `FrozenLake-v1` setup from `An_Introduction_to_Q_Learning.ipynb`.

The `A2CAgent` training algorithm below is copied unchanged from `RL_A2C.ipynb`. The adaptation happens outside the algorithm: FrozenLake's discrete state is converted to a one-hot vector, and the actor-critic network uses a lazy first layer so it can accept that vector.


## Imports and Seeds


In [1]:
import random
from collections import deque

import gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions.categorical import Categorical
from tqdm import tqdm

SEED = 7

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(SEED)


Additionally, Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


## Environment

FrozenLake exposes observations as integer state ids. The original A2C loop expects a tensor-like observation, so this wrapper converts each state id into a one-hot vector while preserving the legacy Gym `reset()` and `step()` interface used by `RL_A2C.ipynb`.


In [2]:
class OneHotFrozenLake:
    def __init__(self, env):
        self.env = env
        self.action_space = env.action_space
        self.observation_space = env.observation_space
        self.n_states = env.observation_space.n

    def _one_hot(self, state):
        observation = np.zeros(self.n_states, dtype=np.float32)
        observation[int(state)] = 1.0
        return observation

    def reset(self):
        reset_result = self.env.reset()
        state = reset_result[0] if isinstance(reset_result, tuple) else reset_result
        return self._one_hot(state)

    def step(self, action):
        step_result = self.env.step(action)
        if len(step_result) == 5:
            state, reward, terminated, truncated, info = step_result
            done = terminated or truncated
        else:
            state, reward, done, info = step_result
        return self._one_hot(state), reward, done, info

    def close(self):
        self.env.close()


def make_frozenlake_env(seed=None):
    env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False)
    if seed is not None:
        try:
            env.reset(seed=seed)
        except TypeError:
            env.seed(seed)
        env.action_space.seed(seed)
    return OneHotFrozenLake(env)


env = make_frozenlake_env(SEED)
print("State space:", env.observation_space.n)
print("Action space:", env.action_space.n)
print("First one-hot observation:", env.reset())


State space: 16
Action space: 4
First one-hot observation: [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


## Neural Network

Only the first layer differs from the CartPole notebook: `nn.LazyLinear` infers the FrozenLake one-hot observation size on the first forward pass. The actor and critic heads stay the same.


In [ ]:
class ActorCritic(nn.Module):
    def __init__(self, hidden_size, num_outputs):
        super(ActorCritic, self).__init__()
        self.hidden_layer = nn.LazyLinear(hidden_size)
        self.actor_layer = nn.Linear(hidden_size, num_outputs)
        self.critic_layer = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = F.relu(self.hidden_layer(x))
        action_probs = F.softmax(self.actor_layer(x), dim=-1)
        value = self.critic_layer(x)
        return action_probs, value


## A2C

The next cell is the original `A2CAgent` class from `RL_A2C.ipynb`.


In [ ]:
class A2CAgent:
    def __init__(self, env, num_episodes=1000, max_steps=500, gamma=0.99, lr=1e-3, hidden_size=256):
        self.env = env
        self.num_episodes = int(num_episodes)
        self.max_steps = max_steps
        self.gamma = gamma
        self.lr = lr
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # define your actor critic network, its optimzer and the loss function
        self.policy_net = ActorCritic(hidden_size, env.action_space.n).to(self.device)
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=self.lr)
        self.critic_loss = nn.MSELoss()

    def choose_action(self, state):
        # choose an action by sampling from the policy distribution
        state = torch.FloatTensor(state).to(self.device)
        action_probs, _ = self.policy_net(state)
        action = torch.multinomial(action_probs, 1).item()
        return action

    def compute_returns(self, rewards):
        # calculate the discounted rewards used in the training process
        R = 0
        returns = []
        for r in reversed(rewards):
            R *= self.gamma
            R += r
            returns.insert(0, R)
        returns = torch.tensor(returns).to(self.device)
        return returns

    def train(self):
        episode_rewards = np.array([])
        with tqdm(range(self.num_episodes)) as pbar:
            for episode in pbar:
                # Implement the training loop
                state = self.env.reset()
                episode_reward = 0
                values = []
                rewards = []
                logits = []

                for step in range(self.max_steps):
                    # first you need to gather log probabilities, state values and rewards from a trajectory
                    # update the 'episode_reward'
                    state = torch.FloatTensor(state).to(self.device)
                    action_probs, value = self.policy_net(state)
                    action = torch.multinomial(action_probs, 1).item()

                    next_state, reward, done, _ = self.env.step(action)
                    new_probs = torch.log(action_probs[action])
                    values.append(value)
                    rewards.append(reward)
                    logits.append(new_probs)

                    episode_reward += reward
                    state = next_state
                    #terminate state
                    if done:
                        break


                episode_rewards = np.append(episode_rewards, episode_reward)

                # calculate the discounted rewards
                returns = self.compute_returns(rewards)

                # calculate advantage
                values = torch.cat(values)
                logits = torch.stack(logits)
                advantage = returns - values

                # compute actor and critic losses
                actorLoss = -(logits * advantage.detach()).mean()
                criticLoss = self.critic_loss(values, returns)
                lossEval = actorLoss + criticLoss

                self.optimizer.zero_grad()
                lossEval.backward()
                self.optimizer.step()

                pbar.set_description(f"Episode {episode}, Reward: {np.mean(episode_reward)}")

        self.env.close()
        return episode_rewards


## Train


In [ ]:
set_seed(SEED)

env = make_frozenlake_env(SEED)
num_episodes = 2000
max_steps = 99
gamma = 0.99
lr = 3e-3
hidden_size = 128

a2c_model = A2CAgent(
    env,
    num_episodes=num_episodes,
    max_steps=max_steps,
    gamma=gamma,
    lr=lr,
    hidden_size=hidden_size,
)

rewards = a2c_model.train()
print(f"Mean training reward over last 100 episodes: {rewards[-100:].mean():.2f}")


## Evaluation

The evaluation uses a greedy action (`argmax`) from the learned policy. On deterministic non-slippery FrozenLake, a correct policy should get reward `1.0` in every episode.


In [ ]:
ACTION_NAMES = np.array(["LEFT", "DOWN", "RIGHT", "UP"])
LAKE_MAP = np.array([
    list("SFFF"),
    list("FHFH"),
    list("FFFH"),
    list("HFFG"),
])


def greedy_action(agent, observation):
    device = next(agent.policy_net.parameters()).device
    observation_tensor = torch.tensor(observation, dtype=torch.float32).to(device)
    with torch.no_grad():
        action_probs, _ = agent.policy_net(observation_tensor)
    return int(action_probs.argmax().item())


def evaluate_agent(agent, n_eval_episodes=100, max_steps=99):
    env = make_frozenlake_env()
    episode_rewards = []
    episode_paths = []

    for _ in range(n_eval_episodes):
        observation = env.reset()
        total_reward = 0
        path = []

        for _ in range(max_steps):
            action = greedy_action(agent, observation)
            path.append(action)
            observation, reward, done, _ = env.step(action)
            total_reward += reward
            if done:
                break

        episode_rewards.append(total_reward)
        episode_paths.append(path)

    env.close()
    return np.mean(episode_rewards), np.std(episode_rewards), episode_paths[0]


def policy_grid(agent):
    device = next(agent.policy_net.parameters()).device
    states = torch.eye(16, dtype=torch.float32).to(device)
    with torch.no_grad():
        action_probs, _ = agent.policy_net(states)
    actions = action_probs.argmax(dim=1).cpu().numpy().reshape(4, 4)

    grid = ACTION_NAMES[actions]
    grid = grid.astype(object)
    grid[LAKE_MAP == "H"] = "HOLE"
    grid[LAKE_MAP == "G"] = "GOAL"
    return grid


mean_reward, std_reward, example_path = evaluate_agent(a2c_model)
print(f"Mean_reward={mean_reward:.2f} +/- {std_reward:.2f}")
print("Example greedy path:", " -> ".join(ACTION_NAMES[example_path]))
print("Learned greedy policy grid:")
print(policy_grid(a2c_model))

assert mean_reward == 1.0, "A2C did not learn a goal-reaching greedy policy. Try a different seed or more episodes."
